<a href="https://colab.research.google.com/github/Haithm-alameri/OptimizationSystems-Course/blob/main/Lab%2001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Systems Modeling & Tooling (Optimization Systems)

**Course:** Optimization Systems (Graduate)  
**Lecture:** 1 — Systems Modeling & Tooling  

**Goals:**

- Get a working Google Colab environment for this course.
- Build and solve a tiny *capacity expansion* model in Pyomo.
- Inspect solver logs and the optimal solution.
- Perform a simple sensitivity experiment (e.g., demand +10%).
- (Optional) See a tiny assignment example in OR-Tools CP-SAT.

> Always run the installation cell **first** when you open this notebook.

In [ ]:
# Install required libraries (if not already present)
!pip install python-pptx matplotlib numpy

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pptx import Presentation
from pptx.util import Inches
import os

# ------------------------------------------------------------
# 1. Generate all images (same as before, but ensure English labels)
# ------------------------------------------------------------

# ----- 1. Slaughter comparison chart -----
districts = ['Cairo', 'Al-Mudhaffar', 'Salah']
normal = [2500, 1800, 1200]
ramadan = [4000, 2800, 2000]
friday = [5500, 3800, 2700]

x = np.arange(len(districts))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, normal, width, label='Normal Day', color='#66b3ff')
bars2 = ax.bar(x, ramadan, width, label='Ramadan', color='#ffcc99')
bars3 = ax.bar(x + width, friday, width, label='Friday', color='#99ff99')

ax.set_xlabel('District', fontsize=12)
ax.set_ylabel('Number of Chickens per Day', fontsize=12)
ax.set_title('Daily Slaughter Comparison by District', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(districts)
ax.legend()

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height):,}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('slaughter_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 2. Fat & Feather estimates chart -----
categories = ['Fat', 'Feather']
averages = [4.0, 3.5]
lower = [3.5, 3.0]
upper = [4.5, 4.0]
errors = [[avg - low for avg, low in zip(averages, lower)],
          [up - avg for avg, up in zip(averages, upper)]]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(categories, averages, yerr=errors, capsize=5,
              color=['#ff9999', '#66b3ff'], edgecolor='black', linewidth=1)

ax.set_ylabel('kg per 100 chickens', fontsize=12)
ax.set_title('Estimated Fat and Feather Quantities\n(per 100 chickens)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 5.5)

for bar, avg in zip(bars, averages):
    ax.annotate(f'{avg:.1f} kg', xy=(bar.get_x() + bar.get_width()/2, avg),
                xytext=(0, 5), textcoords="offset points", ha='center', va='bottom', fontsize=10)

for i, (cat, low, high) in enumerate(zip(categories, lower, upper)):
    ax.text(i, high + 0.2, f'({low}–{high})', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fat_feather_estimates.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 3. Transesterification figure -----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Transesterification of Waste Animal Fats to Biodiesel', fontsize=16, fontweight='bold')

# Left side: equation
ax1.axis('off')
ax1.text(0.05, 0.8, r'$\mathrm{Triglyceride}$', fontsize=12)
ax1.text(0.05, 0.73, r'$\mathrm{CH_2-O-CO-R_1}$', fontsize=10)
ax1.text(0.05, 0.66, r'$\mathrm{CH-O-CO-R_2}$', fontsize=10)
ax1.text(0.05, 0.59, r'$\mathrm{CH_2-O-CO-R_3}$', fontsize=10)

ax1.text(0.5, 0.70, r'$+$', fontsize=14)
ax1.text(0.6, 0.70, r'$\mathrm{3\;CH_3OH}$', fontsize=12)
ax1.text(0.6, 0.64, r'(Methanol)', fontsize=9, style='italic')

ax1.text(0.9, 0.70, r'$\rightarrow$', fontsize=14)

ax1.text(1.1, 0.8, r'$\mathrm{Biodiesel}$', fontsize=12)
ax1.text(1.1, 0.73, r'$\mathrm{R_1-COO-CH_3}$', fontsize=10)
ax1.text(1.1, 0.66, r'$\mathrm{R_2-COO-CH_3}$', fontsize=10)
ax1.text(1.1, 0.59, r'$\mathrm{R_3-COO-CH_3}$', fontsize=10)
ax1.text(1.1, 0.53, r'(Fatty Acid Methyl Esters)', fontsize=9, style='italic')

ax1.text(1.55, 0.70, r'$+$', fontsize=14)
ax1.text(1.7, 0.73, r'$\mathrm{Glycerol}$', fontsize=12)
ax1.text(1.7, 0.66, r'$\mathrm{CH_2-OH}$', fontsize=10)
ax1.text(1.7, 0.59, r'$\mathrm{CH-OH}$', fontsize=10)
ax1.text(1.7, 0.52, r'$\mathrm{CH_2-OH}$', fontsize=10)

# Right side: tables
ax2.axis('off')
ax2.set_title('Optimal Conditions & Biodiesel Specifications', fontsize=14, fontweight='bold')

cond_data = [['Parameter', 'Optimal Value', 'Reference'],
             ['Methanol:Oil molar ratio', '6:1 – 12:1', 'Afifi et al., 2024'],
             ['Catalyst (NaOH/KOH)', '1% wt', 'Chakraborty & Sahu, 2014'],
             ['Temperature', '50 – 70 °C', 'Banković-Iljić et al., 2014'],
             ['Reaction time', '1 – 3 hours', 'Mata et al., 2011'],
             ['Expected yield', '85 – 95%', 'Afifi et al., 2024']]
table1 = ax2.table(cellText=cond_data, loc='upper left', cellLoc='center',
                   colWidths=[0.3, 0.3, 0.3], bbox=[0.0, 0.55, 0.95, 0.4])
table1.auto_set_font_size(False)
table1.set_fontsize(9)
for (i, j), cell in table1.get_celld().items():
    if i == 0:
        cell.set_facecolor('#d3d3d3')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

spec_data = [['Property', 'Specification', 'Standard'],
             ['Viscosity @ 40°C', '1.9 – 6.0 mm²/s', 'ASTM D6751'],
             ['Density @ 15°C', '860 – 900 kg/m³', 'EN 14214'],
             ['Flash point', '> 93 °C', 'ASTM D6751']]
table2 = ax2.table(cellText=spec_data, loc='upper left', cellLoc='center',
                   colWidths=[0.3, 0.35, 0.25], bbox=[0.0, 0.15, 0.95, 0.35])
table2.auto_set_font_size(False)
table2.set_fontsize(9)
for (i, j), cell in table2.get_celld().items():
    if i == 0:
        cell.set_facecolor('#d3d3d3')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

ax2.text(0.05, 0.05, 'References: ASTM D6751, EN 14214', fontsize=8, style='italic', transform=ax2.transAxes, alpha=0.7)

plt.tight_layout()
plt.savefig('transesterification.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 4. Feather pathway figure -----
fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 12)
ax.set_ylim(0, 9)
ax.axis('off')
ax.text(6, 8.6, "Feather Valorization Pathway: Keratin → Organic Fertilizer", fontsize=16, fontweight='bold', ha='center')

def add_box(x, y, w, h, title, subtitle=None, color='#f0f0f0'):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.2, title, fontweight='bold', ha='center')
    if subtitle:
        ax.text(x + w/2, y + h - 0.45, subtitle, ha='center', fontsize=8)

add_box(1, 7.5, 2.5, 0.8, "Source", "Poultry feathers from slaughtering\n(Polymers, 2024)")
add_box(4.5, 7.5, 3.5, 0.8, "Properties", "Keratin 85–90% • Nitrogen 10–15%\n(Hadas & Kautsky, 1994; Mézes et al., 2015)")
ax.annotate('', xy=(4.4, 7.9), xytext=(3.6, 7.9), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

add_box(1, 5.8, 3, 0.9, "Step 1: Collection & Pre‑treatment", "Separate feathers, chop/grind\n(Ind. J. Vet. & Anim. Sci. Res., 2017)")
add_box(1, 4.0, 3, 1.0, "Step 2: Aerobic Composting", "Mix with carbon source (C/N 20:1–30:1, moisture 50–60%)\nTurn periodically, 4–8 weeks\n(Mézes et al., 2015; Frontiers in Microbiology, 2022)")
add_box(1, 2.2, 3, 0.9, "Step 3: Maturation", "Fully decomposed, pathogen‑free, nitrogen‑rich\n(Hadas & Kautsky, 1994)")
ax.annotate('', xy=(2.5, 5.7), xytext=(2.5, 5.0), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
ax.annotate('', xy=(2.5, 3.9), xytext=(2.5, 3.1), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

add_box(6.5, 4.2, 4, 1.8, "Final Output", "Organic Fertilizer\n• Improves soil fertility\n• Supports sustainable agriculture\n• Slow‑release nitrogen\n(Al‑Rahmi et al., 2025)", color='#c8e6c9')
ax.annotate('', xy=(6.4, 5.0), xytext=(4.1, 2.6), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

ax.text(0.5, 0.2, "References: Hadas & Kautsky, 1994; Mézes et al., 2015; Polymers, 2024; Frontiers in Microbiology, 2022; Al‑Rahmi et al., 2025",
        fontsize=7, style='italic', ha='left', alpha=0.7)

plt.tight_layout()
plt.savefig('feather_fertilizer.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 5. Biorefinery layout -----
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

ax.text(7, 9.6, "Biorefinery Top‑View Layout (Fat & Feather Processing Lines)", fontsize=16, fontweight='bold', ha='center')
ax.text(0.5, 9.2, "Plant type: Small‑scale Biorefinery", fontsize=10)
ax.text(0.5, 9.0, "Capacity: 1–2 tons waste/day (Mohan et al., 2025)", fontsize=9, style='italic')

def add_box_layout(x, y, w, h, title, subtitle=None, color='#e0e0e0'):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.2, title, fontweight='bold', ha='center')
    if subtitle:
        ax.text(x + w/2, y + h - 0.45, subtitle, ha='center', fontsize=8)

# Common units
add_box_layout(0.5, 7.5, 2.5, 1.0, "Reception", "Waste intake")
add_box_layout(3.5, 7.5, 2.5, 1.0, "Storage", "Cooling/Freezing")
add_box_layout(1.5, 5.8, 3.5, 1.0, "Separation Unit", "Manual/Mechanical\nFat / Feather / Offal / Blood", color='#f0f0f0')
ax.annotate('', xy=(3.4, 8.0), xytext=(3.0, 8.0), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(3.25, 6.8), xytext=(3.25, 7.4), arrowprops=dict(arrowstyle='->', lw=1.5))

# Fat line
fat_x = 0.5
add_box_layout(fat_x, 3.8, 2.5, 0.9, "Fat Preparation", "Purification, Heating", color='#ffe6cc')
add_box_layout(fat_x, 2.3, 2.5, 0.9, "Transesterification", "Batch Reactor\n60–70°C, 2–3 h", color='#ffe6cc')
add_box_layout(fat_x, 0.8, 2.5, 0.9, "Purification", "Separation, Washing, Drying", color='#ffe6cc')
add_box_layout(fat_x + 3.0, 0.8, 2.5, 0.9, "Biodiesel", "Storage Tanks", color='#c8e6c9')
# Arrows fat line
ax.annotate('', xy=(fat_x+1.25, 3.7), xytext=(fat_x+1.25, 4.2), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(fat_x+1.25, 2.2), xytext=(fat_x+1.25, 2.7), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(fat_x+1.25, 0.7), xytext=(fat_x+1.25, 1.2), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(fat_x+2.75, 1.25), xytext=(fat_x+2.5, 1.25), arrowprops=dict(arrowstyle='->', lw=1))

# Feather line
feath_x = 8.0
add_box_layout(feath_x, 3.8, 2.5, 0.9, "Feather Preparation", "Chopping / Grinding", color='#ccf0e0')
add_box_layout(feath_x, 2.3, 2.5, 0.9, "Aerobic Composting", "Beds with turning\n4–8 weeks", color='#ccf0e0')
add_box_layout(feath_x, 0.8, 2.5, 0.9, "Screening & Purification", "", color='#ccf0e0')
add_box_layout(feath_x + 3.0, 0.8, 2.5, 0.9, "Organic Fertilizer", "Bagging", color='#c8e6c9')
# Arrows feather line
ax.annotate('', xy=(feath_x+1.25, 3.7), xytext=(feath_x+1.25, 4.2), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(feath_x+1.25, 2.2), xytext=(feath_x+1.25, 2.7), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(feath_x+1.25, 0.7), xytext=(feath_x+1.25, 1.2), arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate('', xy=(feath_x+2.75, 1.25), xytext=(feath_x+2.5, 1.25), arrowprops=dict(arrowstyle='->', lw=1))

# Arrows from separation
ax.annotate('', xy=(fat_x+1.25, 4.5), xytext=(3.25, 5.5), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
ax.annotate('', xy=(feath_x+1.25, 4.5), xytext=(3.25, 5.5), arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
ax.text(3.0, 4.9, "Fat", fontsize=8, ha='center', color='gray')
ax.text(5.5, 4.9, "Feather", fontsize=8, ha='center', color='gray')

# Requirements box
add_box_layout(0.5, 0.15, 13, 0.6, "Basic Requirements: Area 300–400 m² | Electricity, Water, Sanitation | Staff 4–6 workers", color='#f9f9f9')
ax.text(7, 0.45, "Basic Requirements: Area 300–400 m² | Electricity, Water, Sanitation | Staff 4–6 workers", ha='center', fontsize=9)
ax.text(0.5, 0.02, "References: Mohan et al., 2025; Souza, 2022; Afifi et al., 2024; Mata et al., 2011; Mézes et al., 2015",
        fontsize=7, style='italic', ha='left', alpha=0.7)

plt.tight_layout()
plt.savefig('biorefinery_layout.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 6. Economic assumptions table -----
fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
ax.text(0.5, 0.95, "Economic Assumptions for Financial Analysis", fontsize=14, fontweight='bold', ha='center', transform=ax.transAxes)

headers = ['Item', 'Value', 'Source']
rows = [
    ['Operating days/year', '300 days', 'Estimated'],
    ['Fat available (tons/year)', '_____', 'Field study'],
    ['Feather available (tons/year)', '_____', 'Field study'],
    ['Biodiesel price (YER/liter)', '500', 'Barua et al., 2020a'],
    ['Organic fertilizer price (YER/kg)', '50', 'Souza, 2022'],
    ['Fat → Biodiesel conversion rate', '85%', 'Afifi et al., 2024'],
    ['Feather → Fertilizer conversion rate', '80%', 'Mézes et al., 2015']
]

table = ax.table(cellText=rows, colLabels=headers, loc='center',
                 cellLoc='left', colWidths=[0.4, 0.3, 0.3],
                 bbox=[0.1, 0.1, 0.8, 0.7])
table.auto_set_font_size(False)
table.set_fontsize(10)
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#d3d3d3')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

ax.text(0.5, 0.05, "Note: Quantities (_____) will be filled after field study completion.",
        fontsize=9, style='italic', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.01, "References: Barua et al., 2020a; Souza, 2022; Afifi et al., 2024; Mézes et al., 2015",
        fontsize=7, style='italic', ha='center', transform=ax.transAxes, alpha=0.7)

plt.tight_layout()
plt.savefig('economic_assumptions.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 7. Economic feasibility tables -----
fig, ax = plt.subplots(figsize=(12, 12))
ax.axis('off')
ax.text(0.5, 0.97, "Economic Feasibility – Cost, Revenue & Financial Indicators",
        fontsize=16, fontweight='bold', ha='center', transform=ax.transAxes)

# Costs
cost_headers = ['Item', 'Cost (YER)', 'Notes']
cost_rows = [
    ['INVESTMENT', '', ''],
    ['Main equipment', '_____', 'Reactor, tanks, chopper, composting beds'],
    ['Construction & infrastructure', '_____', 'Area 300–400 m²'],
    ['Other initial costs', '_____', 'Licenses, studies, transport'],
    ['Total investment', '_____', ''],
    ['', '', ''],
    ['ANNUAL OPERATING COSTS', '', ''],
    ['Raw materials (if paid)', '_____', 'Fat, feathers (may be free)'],
    ['Energy (electricity, water)', '_____', ''],
    ['Labor (4–6 workers)', '_____', ''],
    ['Maintenance & operation', '_____', '5% of investment'],
    ['Total annual operating', '_____', '']
]
table1 = ax.table(cellText=cost_rows, colLabels=cost_headers, loc='center',
                  cellLoc='left', colWidths=[0.35, 0.2, 0.4],
                  bbox=[0.1, 0.62, 0.8, 0.32])
table1.auto_set_font_size(False)
table1.set_fontsize(9)
for (i, j), cell in table1.get_celld().items():
    if i == 0 or i == 6:
        cell.set_facecolor('#e0e0e0')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

# Revenues
rev_headers = ['Item', 'Annual Quantity', 'Price (YER)', 'Revenue (YER)']
rev_rows = [
    ['Biodiesel', '_____ L', '500', '_____'],
    ['Organic fertilizer', '_____ kg', '50', '_____'],
    ['Glycerol (by‑product)', '_____ L', '100', '_____'],
    ['TOTAL', '', '', '_____']
]
table2 = ax.table(cellText=rev_rows, colLabels=rev_headers, loc='center',
                  cellLoc='left', colWidths=[0.25, 0.2, 0.2, 0.3],
                  bbox=[0.1, 0.4, 0.8, 0.18])
table2.auto_set_font_size(False)
table2.set_fontsize(9)
for (i, j), cell in table2.get_celld().items():
    if i == 0:
        cell.set_facecolor('#e0e0e0')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

# Indicators
ind_headers = ['Indicator', 'Value', 'Reference']
ind_rows = [
    ['Net annual cash flow', '_____', 'Revenues – Operating costs'],
    ['Payback period', '_____ years', 'Blank & Tarquin, 2018'],
    ['NPV @ 10%', '_____ YER', ''],
    ['IRR', '_____ %', 'Barua et al., 2020a']
]
table3 = ax.table(cellText=ind_rows, colLabels=ind_headers, loc='center',
                  cellLoc='left', colWidths=[0.4, 0.3, 0.3],
                  bbox=[0.1, 0.22, 0.8, 0.14])
table3.auto_set_font_size(False)
table3.set_fontsize(9)
for (i, j), cell in table3.get_celld().items():
    if i == 0:
        cell.set_facecolor('#e0e0e0')
        cell.set_text_props(weight='bold')
    cell.set_edgecolor('black')

# Footnotes
ax.text(0.5, 0.05, "Note: All values are placeholders (_____) and will be updated after field study and lab experiments.\n"
                   "Financial indicators will be calculated using Excel and a screenshot will be added.",
        fontsize=9, style='italic', ha='center', transform=ax.transAxes)
ax.text(0.5, 0.02, "References: Blank & Tarquin, 2018; Barua et al., 2020a; Afifi et al., 2024; Mézes et al., 2015",
        fontsize=7, style='italic', ha='center', transform=ax.transAxes, alpha=0.7)

plt.tight_layout()
plt.savefig('economic_feasibility.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 8. Expected outcomes -----
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.text(5, 7.6, "Expected Outcomes of the Research", fontsize=16, fontweight='bold', ha='center')

frame = FancyBboxPatch((0.5, 0.5), 9, 6.5, boxstyle="round,pad=0.2",
                       facecolor='none', edgecolor='#cccccc', linewidth=1.5)
ax.add_patch(frame)

outcomes = [
    ("•  Documented database of waste quantities in Taiz", "(Hemavarshini et al., 2025)"),
    ("•  Optimized laboratory methodology for biodiesel and fertilizer", "(Afifi et al., 2024; Hadas & Kautsky, 1994)"),
    ("•  Feasible engineering design of an integrated biorefinery", "(Mohan et al., 2025)"),
    ("•  Realistic economic feasibility study with sensitivity analysis", "(Blank & Tarquin, 2018; Barua et al., 2020a)"),
    ("•  Recommendations for authorities and private sector", "(Mohan et al., 2025; Sustainability, 2019)")
]

y = 6.7
for text, ref in outcomes:
    ax.text(1.2, y, text, fontsize=11, fontweight='bold', va='center')
    ax.text(1.2, y - 0.25, ref, fontsize=9, style='italic', va='center')
    y -= 1.0

ax.text(5, 0.8, "All outcomes align with research objectives and support sustainable waste management.",
        fontsize=9, ha='center', style='italic', color='gray')

plt.tight_layout()
plt.savefig('expected_outcomes.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 9. Timeline -----
phases = ["Field data collection", "Lab experiments (fat + feather)", "Engineering design", "Economic model & feasibility", "Thesis writing (parallel)"]
durations = [4, 7, 4, 3, 0]
starts = [0, 4, 11, 15, 0]
colors = ['#66b3ff', '#ffcc99', '#99ff99', '#ff9999', '#c0c0c0']

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 19)
ax.set_ylim(0, len(phases) + 0.5)

for i, (phase, dur, start, color) in enumerate(zip(phases, durations, starts, colors)):
    if dur > 0:
        ax.barh(i+1, dur, left=start, height=0.6, color=color, edgecolor='black', linewidth=1)
        ax.text(start + dur + 0.2, i+1, f"{dur} weeks", va='center', fontsize=9)

ax.hlines(y=5, xmin=0, xmax=18, colors='gray', linestyles='dashed', linewidth=2)
ax.text(18.2, 5, "Thesis writing (parallel)", va='center', fontsize=9, style='italic')

ax.set_yticks(np.arange(1, len(phases)+1))
ax.set_yticklabels(phases)
ax.set_xlabel("Weeks from start", fontsize=11)
ax.set_title("Research Timeline (5–6 months)", fontsize=14, fontweight='bold')

total_weeks = sum(durations[:4])
ax.text(0.5, -0.8, f"Total active phases: {total_weeks} weeks (≈ {total_weeks/4:.1f} months).\n"
                   f"Thesis writing runs in parallel, overall duration ≈ 5–6 months.",
        transform=ax.transAxes, fontsize=9, style='italic', ha='left')

plt.tight_layout()
plt.savefig('timeline.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 10. Conclusion -----
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')
ax.text(5, 6.6, "Conclusion", fontsize=18, fontweight='bold', ha='center')

frame = FancyBboxPatch((0.5, 0.4), 9, 5.5, boxstyle="round,pad=0.2",
                       facecolor='none', edgecolor='#cccccc', linewidth=1.5)
ax.add_patch(frame)

points = [
    ("•  The project offers an integrated solution to an environmental and economic problem in Taiz,",
     "based on circular economy principles (Mohan et al., 2025)."),
    ("•  It relies on locally available waste using globally proven technologies",
     "(Banković-Iljić et al., 2014; Polymers, 2024)."),
    ("•  Expected outcomes include pollution reduction, local energy production,",
     "and support for sustainable agriculture (Al‑Rahmi et al., 2025).")
]

y = 5.8
for line1, line2 in points:
    ax.text(1.2, y, line1, fontsize=10, va='center')
    ax.text(1.2, y - 0.25, line2, fontsize=10, va='center')
    y -= 1.0

request_box = FancyBboxPatch((1.2, 0.8), 7.6, 0.8, boxstyle="round,pad=0.1",
                              facecolor='#fff2cc', edgecolor='#ffcc66', linewidth=1.5)
ax.add_patch(request_box)
ax.text(5, 1.2, "MOTION: APPROVE THE METHODOLOGY AND PROCEED TO FIELD & LABORATORY WORK",
        fontsize=11, fontweight='bold', ha='center', va='center')

plt.tight_layout()
plt.savefig('conclusion.png', dpi=300, bbox_inches='tight')
plt.close()

# ----- 11. Thank you slide -----
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 8)
ax.set_ylim(0, 5)
ax.axis('off')

frame = FancyBboxPatch((0.2, 0.2), 7.6, 4.6, boxstyle="round,pad=0.2",
                       facecolor='#f9f9f9', edgecolor='#cccccc', linewidth=1)
ax.add_patch(frame)

ax.text(4, 3.8, "Thank You", fontsize=28, fontweight='bold', ha='center')
ax.text(4, 3.0, "Questions from the Committee", fontsize=16, ha='center', style='italic')
ax.text(4, 2.2, "Contact:", fontsize=14, fontweight='bold', ha='center')
ax.text(4, 1.8, "haithmameri@gmail.com", fontsize=12, ha='center')
ax.text(4, 1.4, "774435043", fontsize=12, ha='center')
ax.axhline(y=2.5, xmin=0.2, xmax=0.8, linewidth=1, color='#cccccc', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('thank_you.png', dpi=300, bbox_inches='tight')
plt.close()

# ------------------------------------------------------------
# 2. Create PowerPoint presentation (all in English)
# ------------------------------------------------------------
prs = Presentation()

# Helper: add slide with title and bullet points (content layout)
def add_slide_with_bullets(title, bullets):
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = title
    content = slide.placeholders[1]
    tf = content.text_frame
    tf.text = bullets[0]
    for bullet in bullets[1:]:
        p = tf.add_paragraph()
        p.text = bullet
        p.level = 0

# Helper: add slide with an image (centered)
def add_slide_with_image(title, image_path):
    slide = prs.slides.add_slide(prs.slide_layouts[5])  # blank layout
    # Add title as a textbox
    left = Inches(0.5)
    top = Inches(0.2)
    width = Inches(9)
    height = Inches(1)
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.text = title
    tf.paragraphs[0].font.size = 28
    tf.paragraphs[0].font.bold = True
    # Add image
    slide.shapes.add_picture(image_path, Inches(0.5), Inches(1.2), width=Inches(9))

# ----- Slide 1: Title (layout 0) -----
slide = prs.slides.add_slide(prs.slide_layouts[0])
title = slide.shapes.title
title.text = "Valorization of Poultry Slaughterhouse Waste\ninto Biodiesel and Organic Fertilizer"
subtitle = slide.placeholders[1]
subtitle.text = ("Haitham Abdulrahman Al-Amiri – Master of Industrial Engineering\n"
                 "Supervisor: Dr. Mukhtar Ali Al-Omrani\n"
                 "Al-Saeed Faculty – Taiz University\n"
                 "Seminar – Spring Semester ____")

# ----- Slide 2: Problem -----
add_slide_with_bullets("Problem Statement", [
    "Poultry slaughterhouse waste (fat, feathers, offal, blood) accumulates daily in Taiz without proper treatment (Hemavarshini et al., 2025).",
    "It causes soil and groundwater pollution, unpleasant odors, harmful gas emissions, and health risks (Mohan et al., 2025).",
    "The main districts of Taiz lack infrastructure for managing this waste, missing the opportunity to use it as an economic resource."
])

# ----- Slide 3: Research Gap -----
add_slide_with_bullets("Research Gap", [
    "Locally: No previous studies in Yemen on estimating waste quantities or converting them into valuable products; absence of documented field data.",
    "Globally: Studies focused on large industrial slaughterhouses (Banković-Iljić et al., 2014; Mata et al., 2011) and did not address medium‑sized cities like Taiz. Integrated studies (field + lab + design + feasibility) are limited (Barua et al., 2020a; Ben Jaber et al., 2025).",
    "This research fills the gap by: accurate field quantification, optimized lab experiments, a feasible small‑scale biorefinery design, and a realistic economic model."
])

# ----- Slide 4: Objectives -----
add_slide_with_bullets("Objectives", [
    "Main objective: Develop an integrated model to convert poultry slaughterhouse waste in Taiz into biodiesel and organic fertilizer, assessing technical and economic feasibility according to international standards (ASTM D6751, EN 14214).",
    "1. Estimate daily slaughter and waste quantities (fat, feathers) in three main districts (Cairo, Al-Mudhaffar, Salah) (Barua et al., 2020b).",
    "2. Characterize the physicochemical properties of fat and feathers (Knothe & Razon, 2017; Polymers, 2024).",
    "3. Conduct lab experiments to produce biodiesel from fat via transesterification using suitable catalysts (Afifi et al., 2024; Chakraborty & Sahu, 2014).",
    "4. Conduct lab experiments to convert feathers into organic fertilizer through aerobic composting (Hadas & Kautsky, 1994; Mézes et al., 2015).",
    "5. Design a small‑scale biorefinery based on circular economy principles (Mohan et al., 2025; Souza, 2022).",
    "6. Perform an initial economic analysis to calculate financial viability and payback period (Blank & Tarquin, 2018; Barua et al., 2020a)."
])

# ----- Slide 5: Methodology & Estimates (two images side by side) -----
slide5 = prs.slides.add_slide(prs.slide_layouts[5])
# Title
left = Inches(0.5); top = Inches(0.2); width = Inches(9); height = Inches(1)
txBox = slide5.shapes.add_textbox(left, top, width, height)
txBox.text_frame.text = "Methodology and Preliminary Data"
txBox.text_frame.paragraphs[0].font.size = 28
txBox.text_frame.paragraphs[0].font.bold = True
# Add two images
slide5.shapes.add_picture('slaughter_comparison.png', Inches(0.5), Inches(1.2), width=Inches(4.5))
slide5.shapes.add_picture('fat_feather_estimates.png', Inches(5.2), Inches(1.2), width=Inches(4.5))

# ----- Slides 6-14: remaining images -----
add_slide_with_image("Fat Pathway – Transesterification", 'transesterification.png')
add_slide_with_image("Feather Pathway – Conversion to Organic Fertilizer", 'feather_fertilizer.png')
add_slide_with_image("Engineering Design – Biorefinery Layout", 'biorefinery_layout.png')
add_slide_with_image("Economic Model – Assumptions", 'economic_assumptions.png')
add_slide_with_image("Economic Feasibility – Figures", 'economic_feasibility.png')
add_slide_with_image("Expected Outcomes", 'expected_outcomes.png')
add_slide_with_image("Timeline", 'timeline.png')
add_slide_with_image("Conclusion", 'conclusion.png')
add_slide_with_image("Thank You", 'thank_you.png')

# Save presentation
prs.save('seminar_presentation.pptx')
print("✅ Presentation saved as seminar_presentation.pptx")

In [ ]:
# === Install required packages (run this cell first) ===
!pip install -q "protobuf<6" pyomo ortools

## 1. Imports and Data

We will define a tiny capacity expansion instance using pandas DataFrames.

We have:
- A set of **facilities** with fixed costs and capacities.
- A set of **regions** with demands.
- A **shipping cost** from each facility to each region.

In [ ]:
import pandas as pd

from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint,
    NonNegativeReals, Binary, minimize, value, Set
)
from pyomo.opt import SolverFactory

In [ ]:
# --- Toy data inline (you could later replace with CSVs) ---

facilities = pd.DataFrame({
    "id": ["F1", "F2", "F3"],
    "fixed_cost": [5000, 4000, 4500],
    "capacity": [120, 100, 110],
})

regions = pd.DataFrame({
    "id": ["R1", "R2", "R3"],
    "demand": [80, 90, 70],
})

ship_cost = pd.DataFrame(
    [
        ("F1", "R1", 2), ("F1", "R2", 3), ("F1", "R3", 4),
        ("F2", "R1", 3), ("F2", "R2", 2), ("F2", "R3", 3),
        ("F3", "R1", 4), ("F3", "R2", 3), ("F3", "R3", 2),
    ],
    columns=["facility_id", "region_id", "unit_cost"]
)

facilities, regions, ship_cost

(   id  fixed_cost  capacity
 0  F1        5000       120
 1  F2        4000       100
 2  F3        4500       110,
    id  demand
 0  R1      80
 1  R2      90
 2  R3      70,
   facility_id region_id  unit_cost
 0          F1        R1          2
 1          F1        R2          3
 2          F1        R3          4
 3          F2        R1          3
 4          F2        R2          2
 5          F2        R3          3
 6          F3        R1          4
 7          F3        R2          3
 8          F3        R3          2)

## 2. Pyomo Model: Capacity Expansion

We now:

1. Create sets and parameters from the tables.  
2. Create decision variables:
   - `open[f]` ∈ {0,1} — facility open decision
   - `x[f,r]` ≥ 0 — shipped units  
3. Add:
   - Demand satisfaction constraints
   - Capacity linking constraints  
4. Minimize fixed + shipping cost.

In [ ]:
m = ConcreteModel()

m.F = Set(initialize=list(facilities["id"]))
m.R = Set(initialize=list(regions["id"]))

fixed = {row.id: float(row.fixed_cost) for _, row in facilities.iterrows()}
cap = {row.id: float(row.capacity) for _, row in facilities.iterrows()}
dem = {row.id: float(row.demand) for _, row in regions.iterrows()}
c = {(r.facility_id, r.region_id): float(r.unit_cost) for _, r in ship_cost.iterrows()}

m.open = Var(m.F, within=Binary)
m.x = Var(m.F, m.R, within=NonNegativeReals)

def obj_rule(m):
    return sum(fixed[f] * m.open[f] for f in m.F) + \
           sum(c[(f, r)] * m.x[(f, r)] for f in m.F for r in m.R)

m.OBJ = Objective(rule=obj_rule, sense=minimize)

def demand_rule(m, r):
    return sum(m.x[(f, r)] for f in m.F) >= dem[r]
m.Demand = Constraint(m.R, rule=demand_rule)

def capacity_rule(m, f):
    return sum(m.x[(f, r)] for r in m.R) <= cap[f] * m.open[f]
m.Capacity = Constraint(m.F, rule=capacity_rule)

m.pprint()

2 Set Declarations
    F : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'F1', 'F2', 'F3'}
    R : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'R1', 'R2', 'R3'}

2 Var Declarations
    open : Size=3, Index=F
        Key : Lower : Value : Upper : Fixed : Stale : Domain
         F1 :     0 :  None :     1 : False :  True : Binary
         F2 :     0 :  None :     1 : False :  True : Binary
         F3 :     0 :  None :     1 : False :  True : Binary
    x : Size=9, Index=F*R
        Key          : Lower : Value : Upper : Fixed : Stale : Domain
        ('F1', 'R1') :     0 :  None :  None : False :  True : NonNegativeReals
        ('F1', 'R2') :     0 :  None :  None : False :  True : NonNegativeReals
        ('F1', 'R3') :     0 :  None :  None : False :  True : NonNegativeReals
        ('F2', 'R1') :     0 :  None :  None : False

## 3. Solve the Model and Inspect the Solution

We will use the open-source **HiGHS** solver (via `highspy`) on Colab.

If the solver fails for any reason, note the error and discuss with your instructor.

In [ ]:
solver = SolverFactory("highs")
results = solver.solve(m, tee=True)  # tee=True prints solver log

print("\n===== Solution Summary =====")
print("Status:", results.solver.status)
print("Termination condition:", results.solver.termination_condition)

print("Total cost:", value(m.OBJ))
print("Opened facilities:", [f for f in m.F if m.open[f].value > 0.5])

print("\nShipments (only positive flows):")
for f in m.F:
    for r in m.R:
        val = m.x[(f, r)].value
        if val and val > 1e-6:
            print(f"  {f} -> {r}: {val:.2f}")

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 6 rows; 12 cols; 21 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+02]
  Cost    [2e+00, 5e+03]
  Bound   [1e+00, 1e+00]
  RHS     [7e+01, 9e+01]
Presolving model
6 rows, 12 cols, 21 nonzeros  0s
6 rows, 12 cols, 21 nonzeros  0s
Presolve reductions: rows 6(-0); columns 12(-0); nonzeros 21(-0) - Not reduced

Solving MIP model with:
   6 rows
   12 cols (3 binary, 0 integer, 0 implied int., 9 continuous, 0 domain fixed)
   21 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            

### Quick Questions

- Which facilities opened? Why do you think the model chose them?
- Are any facilities open but barely used?
- Which constraints do you expect to be binding at optimum:
  - Capacity constraints?
  - Demand constraints?
  - Both?

## 4. Sensitivity: Demand +10%

Now we increase all demands by 10% and re-solve.

We keep the same model, just change the right-hand side of the demand constraints.

In [ ]:
# Increase demand by 10%
dem_up = {k: v * 1.10 for k, v in dem.items()}
for r in m.R:
    m.Demand[r].set_value(sum(m.x[(f, r)] for f in m.F) >= dem_up[r])

results_up = solver.solve(m, tee=False)

print("===== After +10% demand =====")
print("Status:", results_up.solver.status)
print("Termination condition:", results_up.solver.termination_condition)
print("Total cost:", value(m.OBJ))
print("Opened facilities:", [f for f in m.F if m.open[f].value > 0.5])

===== After +10% demand =====
Status: ok
Termination condition: optimal
Total cost: 14028.0
Opened facilities: ['F1', 'F2', 'F3']


> **Think about:**
> - Did the set of open facilities change?
> - By roughly how much did the total cost increase?
> - Did any new constraints become binding?
>
> These observations will help when we talk about **duality** and **sensitivity analysis** later.

## 5. (Optional) OR-Tools CP-SAT: Tiny Assignment Example

This is a small example of an assignment problem using OR-Tools CP-SAT.

Later in the course we will use CP-SAT more seriously for scheduling and routing.

In [ ]:
from ortools.sat.python import cp_model

cost = [
    [14, 5, 8],
    [7,  8, 3],
    [2,  6, 5],
]

n_workers = len(cost)
n_tasks = len(cost[0])

model = cp_model.CpModel()

x = {}
for i in range(n_workers):
    for j in range(n_tasks):
        x[i, j] = model.NewBoolVar(f"x[{i},{j}]")

# Each worker at most one task
for i in range(n_workers):
    model.Add(sum(x[i, j] for j in range(n_tasks)) <= 1)

# Each task exactly one worker
for j in range(n_tasks):
    model.Add(sum(x[i, j] for i in range(n_workers)) == 1)

model.Minimize(sum(cost[i][j] * x[i, j] for i in range(n_workers) for j in range(n_tasks)))

solver_cp = cp_model.CpSolver()
solver_cp.parameters.max_time_in_seconds = 5

status = solver_cp.Solve(model)
print("Status:", solver_cp.StatusName(status))
print("Objective value:", solver_cp.ObjectiveValue())

for i in range(n_workers):
    for j in range(n_tasks):
        if solver_cp.Value(x[i, j]) == 1:
            print(f"Worker {i} -> Task {j}")

Status: OPTIMAL
Objective value: 10.0
Worker 0 -> Task 1
Worker 1 -> Task 2
Worker 2 -> Task 0


## 6. Lecture 1 Deliverable — Short Memo on Model-Data Contract

Using this notebook and Lecture 1, write a **1-2 page memo** describing the model–data contract for the capacity expansion problem:

- What tables (and columns) does the model expect?
- How do these map to sets, parameters, and constraints?
- What assumptions are you making (demands, capacities, time horizon)?
- How would you structure files and scenarios (e.g., baseline vs +10% demand)?

Follow the memo instructions provided by your instructor.